In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("app") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.ui.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 19:46:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Cell 3
df = spark.read.parquet("data/flights_clean_complete")

In [ ]:
from pyspark.sql.functions import explode

df.filter(df.numLayovers == 0).select(
    explode("segmentsCabinCode").alias("segmentsCabinCode")
).groupBy("segmentsCabinCode").count().show()

In [ ]:
# distribution of economy vs non economy flights
df.groupBy(df.isBasicEconomy).count().show()

In [ ]:
from pyspark.sql.functions import explode

df.filter(df.isBasicEconomy == True).select(
    explode("segmentsCabinCode").alias("segmentsCabinCode")
).groupBy("segmentsCabinCode").count().show()

In [ ]:
from pyspark.sql.functions import expr

economy_only_df = df.filter(
    expr("forall(segmentsCabinCode, x -> x = 'economy')")
)

In [ ]:
economy_only_df.select("isBasicEconomy", "segmentsCabinCode").sample(0.01).show(500)

In [ ]:
from pyspark.sql.functions import explode

economy_only_df.select(
    explode("segmentsCabinCode").alias("cabin")
).groupBy("cabin").count().show()

In [ ]:
print(f"Flights with all cabins economy: {economy_only_df.count():,}.")

In [ ]:
economy_only_df = economy_only_df.drop("fareBasisCode")

In [ ]:
economy_only_df.printSchema()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType

checks = []

for field in df.schema.fields:
    col = field.name
    dtype = field.dataType
    
    # NULL values
    checks.append(
        F.sum(F.when(F.col(col).isNull(), 1).otherwise(0)).alias(f"{col}_nulls")
    )
    
    # Empty strings
    if isinstance(dtype, StringType):
        checks.append(
            F.sum(F.when(F.trim(F.col(col)) == "", 1).otherwise(0)).alias(f"{col}_empty_strings")
        )
    
    # Empty arrays
    if isinstance(dtype, ArrayType):
        checks.append(
            F.sum(F.when(F.size(F.col(col)) == 0, 1).otherwise(0)).alias(f"{col}_empty_arrays")
        )
        
        # Arrays containing null elements
        checks.append(
            F.sum(F.when(F.expr(f"exists({col}, x -> x IS NULL)"), 1).otherwise(0))
            .alias(f"{col}_null_elements")
        )

summary = df.agg(*checks)
summary.show(truncate=False)

In [ ]:
economy_only_df.repartition(16).write.mode("overwrite").parquet("data/flights_economy")

In [3]:
df = spark.read.parquet("data/flights_economy_complete")

In [4]:
economy_sample = df.sample(
    fraction=0.011,
    seed=42,
    withReplacement=False
)

In [5]:
economy_sample

DataFrame[legId: string, searchDate: date, flightDate: date, startingAirport: string, destinationAirport: string, numDaysToFlight: int, numLayovers: int, layoverDurationMinutes: int, seatsRemaining: int, travelDuration: int, totalTravelDistance: int, elapsedDays: int, isBasicEconomy: boolean, isRefundable: boolean, isNonStop: boolean, baseFare: double, totalFare: double, segmentsDepartureTime: array<timestamp>, segmentsArrivalTime: array<timestamp>, segmentsArrivalAirportCode: array<string>, segmentsDepartureAirportCode: array<string>, segmentsAirlineName: array<string>, segmentsAirlineCode: array<string>, segmentsAircraft: array<string>, segmentsDurationInMinutes: array<int>, segmentsDistance: array<int>, segmentsCabinCode: array<string>]

In [6]:
economy_sample.repartition(16).write.mode("overwrite").parquet("data/flights_economy_1gb")

26/03/11 19:46:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
import pandas as pd

In [8]:
data = pd.read_parquet("data/flights_economy_1gb")

In [9]:
size_bytes = data.memory_usage(deep=True).sum()
size_gb = size_bytes / (1024**3)

print(f"Pandas dataframe size in memory: {size_gb:.2f} GB")

Pandas dataframe size in memory: 1.06 GB
